# Notebook 02 — Предобработка и аугментация данных

## Что делаем:
1. Загружаем src-модули с Google Drive
2. Применяем предобработку изображений (обрезка, ресайз, YUV, нормализация)
3. Демонстрируем аугментацию
4. Балансируем датасет по углу руля
5. Создаём DataLoader и проверяем батчи
6. Сравниваем распределение ДО и ПОСЛЕ балансировки

In [ ]:
import subprocess
result = subprocess.run(['git', '-C', '/content/drive/MyDrive/diploma', 'pull'], capture_output=True, text=True)
print(result.stdout or result.stderr)

In [ ]:
!pip install -q torch torchvision numpy pandas matplotlib opencv-python scikit-learn tqdm

from google.colab import drive
drive.mount('/content/drive')

import sys, os, json

# Загружаем конфигурацию из notebook 01
config_path = '/content/drive/MyDrive/diploma/config.json'
with open(config_path) as f:
    cfg = json.load(f)

PROJECT_DIR = cfg['project_dir']
DATA_DIR    = cfg['data_dir']
MODELS_DIR  = cfg['models_dir']
OUTPUTS_DIR = cfg['outputs_dir']
CSV_PATH    = cfg['csv_path']
IMG_DIR     = cfg['img_dir']

# Добавляем src/ в путь для импорта наших модулей
src_path = os.path.join(PROJECT_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('✓ Конфигурация загружена')
print(f'  CSV: {CSV_PATH}')

## Шаг 1. Предобработка одного изображения

Покажем что происходит с изображением перед подачей в сеть:
- Обрезка неинформативных зон (небо, капот)
- Изменение размера до 200×66 (стандарт PilotNet)
- Конвертация BGR → YUV
- Нормализация пикселей в [-1, 1]

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import pandas as pd
from dataset import preprocess_image, CROP_TOP, CROP_BOTTOM

# Загружаем случайное изображение
df = pd.read_csv(CSV_PATH, header=None,
                 names=['center','left','right','steering','throttle','reverse','speed'])

sample = df.sample(1, random_state=7).iloc[0]
img_path = os.path.join(IMG_DIR, os.path.basename(str(sample['center']).strip()))

img_bgr = cv2.imread(img_path)
assert img_bgr is not None, f'Изображение не найдено: {img_path}'

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
h = img_bgr.shape[0]

# Промежуточные шаги
img_cropped  = img_bgr[CROP_TOP : h - CROP_BOTTOM, :, :]
img_resized  = cv2.resize(img_cropped, (200, 66))
img_yuv      = cv2.cvtColor(img_resized, cv2.COLOR_BGR2YUV)
img_norm     = preprocess_image(img_bgr)  # готовый результат

fig, axes = plt.subplots(2, 2, figsize=(13, 6))
fig.suptitle('Конвейер предобработки изображения', fontsize=13, fontweight='bold')

steps = [
    (img_rgb,                        f'1. Оригинал\n{img_rgb.shape[1]}×{img_rgb.shape[0]}'),
    (cv2.cvtColor(img_cropped, cv2.COLOR_BGR2RGB),
                                     f'2. Обрезка (убрали небо/капот)\n{img_cropped.shape[1]}×{img_cropped.shape[0]}'),
    (cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB),
                                     f'3. Ресайз до 200×66'),
    ((img_norm + 1.0) / 2.0,         f'4. YUV + нормализация [-1,1]\n(отображено в [0,1])'),
]

for ax, (img, title) in zip(axes.flatten(), steps):
    ax.imshow(np.clip(img, 0, 1) if img.dtype == np.float32 else img)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'preprocessing_steps.png'), dpi=150)
plt.show()

## Шаг 2. Демонстрация аугментации

Аугментация — искусственное расширение датасета.  
Применяем только к обучающей выборке, чтобы не завысить метрики.

In [ ]:
from dataset import augment_flip, augment_brightness, augment_shift

angle = float(sample['steering'])

np.random.seed(42)

img_flip,      angle_flip      = augment_flip(img_bgr.copy(), angle)
img_brightness                 = augment_brightness(img_bgr.copy())
img_shift,     angle_shift     = augment_shift(img_bgr.copy(), angle)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
fig.suptitle('Виды аугментации данных', fontsize=13, fontweight='bold')

def bgr2rgb(img): return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

aug_data = [
    (bgr2rgb(img_bgr),        f'Оригинал\nУгол: {angle:.3f}'),
    (bgr2rgb(img_flip),       f'Отражение (flip)\nУгол: {angle_flip:.3f}'),
    (bgr2rgb(img_brightness), f'Яркость\nУгол: {angle:.3f} (без изм.)'),
    (bgr2rgb(img_shift),      f'Сдвиг (shift)\nУгол: {angle_shift:.3f}'),
]

for ax, (img, title) in zip(axes, aug_data):
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'augmentation_examples.png'), dpi=150)
plt.show()

## Шаг 3. Балансировка датасета

**Проблема:** ~80% кадров — прямая езда (угол ≈ 0).  
**Решение:** Ограничиваем максимальное количество кадров в каждом интервале углов.

In [ ]:
from dataset import load_samples_from_csv, balance_samples

# Загружаем все сэмплы (×3 камеры)
all_samples = load_samples_from_csv(CSV_PATH, data_dir=os.path.dirname(CSV_PATH))

# Балансируем
balanced_samples = balance_samples(all_samples, bins=25, max_per_bin=400)

# Сравниваем распределения ДО и ПОСЛЕ
angles_before = [s[1] for s in all_samples]
angles_after  = [s[1] for s in balanced_samples]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Балансировка датасета по углу руля', fontsize=13, fontweight='bold')

for ax, angles, title, color in zip(
    axes,
    [angles_before, angles_after],
    ['ДО балансировки', 'ПОСЛЕ балансировки'],
    ['salmon', 'mediumseagreen']
):
    ax.hist(angles, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5)
    straight_pct = (np.abs(np.array(angles)) < 0.05).mean() * 100
    ax.set_title(f'{title}\n{len(angles)} сэмплов | прямая: {straight_pct:.1f}%', fontsize=11)
    ax.set_xlabel('Угол руля')
    ax.set_ylabel('Количество')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'balance_comparison.png'), dpi=150)
plt.show()

## Шаг 4. Создание DataLoader и проверка

In [ ]:
import torch
from dataset import get_dataloaders

torch.manual_seed(42)

# num_workers=0 важно для Colab (многопоточность там нестабильна)
train_loader, val_loader, test_loader = get_dataloaders(
    csv_path=CSV_PATH,
    data_dir=os.path.dirname(CSV_PATH),
    batch_size=32,
    balance=True,
    num_workers=0,
)

# Проверяем форму одного батча
images, angles = next(iter(train_loader))
print(f'Форма батча изображений: {images.shape}')  # (32, 3, 66, 200)
print(f'Форма батча углов:       {angles.shape}')  # (32, 1)
print(f'Диапазон пикселей:       [{images.min():.2f}, {images.max():.2f}]')
print(f'Диапазон углов:          [{angles.min():.3f}, {angles.max():.3f}]')
print(f'\nРазмеры выборок:')
print(f'  Train батчей: {len(train_loader)}')
print(f'  Val батчей:   {len(val_loader)}')
print(f'  Test батчей:  {len(test_loader)}')

In [ ]:
# Показываем примеры из train DataLoader
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
fig.suptitle('Примеры из Train DataLoader (после аугментации и предобработки)',
             fontsize=12, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    if i >= len(images): break
    img = images[i].numpy().transpose(1, 2, 0)
    img = (img + 1.0) / 2.0  # денормализация
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(f'Угол: {angles[i].item():.3f}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'dataloader_samples.png'), dpi=150)
plt.show()
print('\n✓ DataLoader работает корректно')

## Итог

✅ Предобработка изображений реализована (обрезка → ресайз → YUV → нормализация)  
✅ 3 вида аугментации (flip, яркость, сдвиг)  
✅ Балансировка датасета устранила перекос в сторону прямой езды  
✅ DataLoader создан и проверен  

**Следующий шаг:** `03_model_training.ipynb` — обучение модели PilotNet

In [ ]:
# Сохраняем пути для следующего ноутбука
import json
config_path = os.path.join(PROJECT_DIR, 'config.json')
with open(config_path) as f:
    cfg = json.load(f)
# Конфиг уже актуален, просто подтверждаем
print('✓ Конфигурация актуальна:', config_path)